# Notebook 4: Experiment 4 — Multi-Stock Training (80/20)## A Comparative Analysis of BiLSTM and BiGRU for Stock Price Prediction**Experiment:** Train on 3 stocks combined, predict on the remaining 1 stock.  **Train/Test Split:** 80/20 (chronological)  **Models:** BiLSTM, BiGRU, LSTM, GRU  **Scaler:** ProportionScaler (÷ 10,501)  **Metrics:** MSE, RMSE, MAE, MAPE, R² Score  **Combinations:**- Train TLKM+BBCA+ASII → Predict UNVR- Train TLKM+BBCA+UNVR → Predict ASII- Train TLKM+ASII+UNVR → Predict BBCA- Train BBCA+ASII+UNVR → Predict TLKM

In [ ]:
import sys, osimport numpy as npimport pandas as pdimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltimport warningswarnings.filterwarnings('ignore')sys.path.insert(0, '.')from stock_prediction_utils import *set_seed()set_ieee_style()DATA_DIR = '.'TRAIN_RATIO = 0.8RATIO_LABEL = '80_20'EXP_LABEL = f'Exp4_{RATIO_LABEL}'os.makedirs(f'figures/{EXP_LABEL}', exist_ok=True)os.makedirs(f'models/{EXP_LABEL}', exist_ok=True)os.makedirs('results', exist_ok=True)print(f"Experiment 4 - Multi-Stock Training (80/20)")

In [ ]:
# Load all daily dataprint("Loading daily data...")daily_data = load_all_daily_data(DATA_DIR)print("\nAll daily data loaded!")

## Define Training Combinations

In [ ]:
# ============================================================# MULTI-STOCK COMBINATIONS# ============================================================# Each entry: (training_stocks, target_stock)combinations = []for target in STOCKS:    train_stocks = [s for s in STOCKS if s != target]    combinations.append((train_stocks, target))    print(f"  Train: {', '.join(train_stocks)} -> Predict: {target}")

## Run All Multi-Stock Experiments

In [ ]:
# ============================================================# EXPERIMENT 4: Multi-stock training# ============================================================all_results = []all_predictions = {}for train_stocks, target_stock in combinations:    train_label = '+'.join(train_stocks)    pair_key = (train_label, target_stock)        print(f"\n{'#'*60}")    print(f"# TRAIN: {train_label} -> TARGET: {target_stock}")    print(f"{'#'*60}")        # Prepare multi-stock data    train_dfs = [daily_data[s] for s in train_stocks]    test_df = daily_data[target_stock]        X_train, y_train, X_test, y_test, test_dates = prepare_multi_stock_data(        train_dfs, test_df,        train_ratio=TRAIN_RATIO, lookback=LOOKBACK    )    print(f"  X_train (combined): {X_train.shape}, X_test: {X_test.shape}")        all_predictions[pair_key] = {}        for model_type in MODEL_TYPES:        exp_name = f'{EXP_LABEL}_train_{train_label}_target_{target_stock}'                y_true_inv, y_pred_inv, metrics, history = train_and_evaluate(            model_type=model_type,            X_train=X_train, y_train=y_train,            X_test=X_test, y_test=y_test,            experiment_name=exp_name,            save_dir=f'models/{EXP_LABEL}',            epochs=EPOCHS, batch_size=BATCH_SIZE        )                result = {            'Train_Stocks': train_label,            'Target_Stock': target_stock,            'Model': model_type,            **metrics        }        all_results.append(result)        all_predictions[pair_key][model_type] = (y_true_inv, y_pred_inv, test_dates)                plot_actual_vs_predicted(            test_dates, y_true_inv, y_pred_inv,            model_type, f'Train_{train_label}_Target_{target_stock}',            EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'        )print("\n\nAll Experiment 4 (80/20) training complete!")

## Results Summary

In [ ]:
# ============================================================# RESULTS TABLE# ============================================================results_df = pd.DataFrame(all_results)print_results_table(results_df, f"Experiment 4 - Multi-Stock Training (80/20)")results_df.to_csv(f'results/{EXP_LABEL}_results.csv', index=False)print(f"Results saved to results/{EXP_LABEL}_results.csv")

## Visualizations

In [ ]:
# ============================================================# COMPARISON PLOTS# ============================================================for (train_label, target_stock), preds_dict in all_predictions.items():    y_true = preds_dict[MODEL_TYPES[0]][0]    dates = preds_dict[MODEL_TYPES[0]][2]    preds = {mt: preds_dict[mt][1] for mt in MODEL_TYPES if mt in preds_dict}        plot_all_models_comparison(        dates, y_true, preds,        f'Train_{train_label}_Target_{target_stock}',        EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'    )# Metrics bar chartfor metric in ['RMSE', 'MAE', 'MAPE (%)', 'R2']:    plot_metrics_comparison_bar(        results_df, metric, EXP_LABEL,        group_col='Target_Stock', save_dir=f'figures/{EXP_LABEL}'    )print("All visualizations saved!")

In [ ]:
# ============================================================# SUMMARY# ============================================================print("\n" + "="*70)print("  BEST MODEL PER TARGET STOCK (by RMSE)")print("="*70)for target in STOCKS:    target_data = results_df[results_df['Target_Stock'] == target]    if target_data.empty:        continue    best_idx = target_data['RMSE'].idxmin()    best = target_data.loc[best_idx]    print(f"  Target {target}: Trained on {best['Train_Stocks']} + {best['Model']} "          f"(RMSE={best['RMSE']:.4f}, R²={best['R2']:.6f})")